# Day 7: Mode Detection and Clustering
**Date:** Monday 29 June 2026

**Conceptual frame:** Clustering is a claim about structure. The clusters you find
depend on the features and distance metric you choose — which means they depend on
your theory of what makes melodies similar.


```{admonition} Conceptual check — before you code
:class: tip

Answer the self-assessment questions for Day 7 before running the cells below.
Questions open in a new tab — come back here when you're done.

**[→ Open Day 7 Quiz](../quizpages/day7_quiz.md)**
```


In [ ]:
import requests,zipfile
from pathlib import Path
from collections import Counter
from itertools import islice
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from music21 import converter,note,interval
sns.set_theme(style='whitegrid',font_scale=1.1)
plt.rcParams['figure.figsize']=(10,4)
print('OK')

In [ ]:
CORPUS_DIR=Path('beregovski_corpus');KERN_DIR=CORPUS_DIR/'kern'
if not(KERN_DIR.exists() and list(KERN_DIR.glob('*.krn'))):
    CORPUS_DIR.mkdir(exist_ok=True)
    r=requests.get('https://github.com/shanahdt/mode_in_klezmer/archive/refs/heads/main.zip')
    zp=CORPUS_DIR/'repo.zip';zp.write_bytes(r.content)
    import shutil
    with zipfile.ZipFile(zp) as z:z.extractall(CORPUS_DIR)
    src=list(CORPUS_DIR.glob('mode_in_klezmer-*/kern'))
    if src:
        if KERN_DIR.exists():shutil.rmtree(KERN_DIR)
        shutil.copytree(src[0],KERN_DIR);zp.unlink()
print(f'{len(list(KERN_DIR.glob("*.krn")))} files')

In [ ]:
def load_corpus(kern_dir=KERN_DIR,verbose=True):
    pc2d={7:1,9:2,11:3,0:4,2:5,4:6,6:7,8:2,10:3,1:4,3:5,5:6}
    records,sdict={},{}
    for i,f in enumerate(sorted(Path(kern_dir).glob('*.krn'))):
        if verbose and i%50==0: print(f'  {i+1}...')
        try:
            s=converter.parse(str(f));ns=[n for n in s.flat.notes if isinstance(n,note.Note)]
            pcs=[n.pitch.pitchClass for n in ns]
            records[f.stem]={'tune_id':f.stem,'n_notes':len(ns),
                'pitches':[n.nameWithOctave for n in ns],'pitch_classes':pcs,
                'scale_degrees':[pc2d.get(p,0) for p in pcs],
                'intervals':[interval.Interval(ns[j],ns[j+1]).semitones for j in range(len(ns)-1)]}
            sdict[f.stem]=s
        except: pass
    if verbose: print(f'Loaded {len(records)}')
    return pd.DataFrame(records.values()),sdict

def get_ngrams(seq,n): return list(zip(*[islice(seq,i,None) for i in range(n)]))
print('ready')

In [ ]:
df,streams=load_corpus()
try:
    meta=pd.read_csv('https://raw.githubusercontent.com/shanahdt/mode_in_klezmer/main/metadata.csv')
    df=df.merge(meta,on='tune_id',how='left');print(f'{len(df)} tunes')
except Exception as e: print(e)

---
## Part 1: Mode Detector


In [ ]:
from scipy.stats import pearsonr

def pc_profile(df_,mode=None):
    s=df_[df_['mode']==mode] if mode and 'mode' in df_.columns else df_
    counts=[0]*12
    for pcs in s['pitch_classes']:
        for pc in pcs: counts[pc]+=1
    t=sum(counts) or 1
    return [c/t for c in counts]

modes=[m for m in ['freygish','raised_fourth','minor','major']
       if 'mode' in df.columns and m in df['mode'].values]
mode_profiles={m:pc_profile(df,m) for m in modes}
print('Profiles built for:',list(mode_profiles.keys()))

In [ ]:
def detect_mode(tune_id,mode_profiles_):
    row=df[df.tune_id==tune_id]
    if row.empty: return None
    pcs=row.iloc[0]['pitch_classes'];counts=[0]*12
    for pc in pcs: counts[pc]+=1
    t=sum(counts) or 1;profile=[c/t for c in counts]
    return max(mode_profiles_,key=lambda m:pearsonr(profile,mode_profiles_[m])[0])

if modes and 'mode' in df.columns:
    df['predicted_mode']=df['tune_id'].apply(lambda tid:detect_mode(tid,mode_profiles))
    acc=(df['mode']==df['predicted_mode']).mean()
    print(f'Accuracy: {acc:.1%}')

In [ ]:
from sklearn.metrics import confusion_matrix,classification_report

if 'predicted_mode' in df.columns and 'mode' in df.columns:
    valid=df[df.mode.notna()&df.predicted_mode.notna()]
    labels=sorted(valid.mode.unique())
    cm=confusion_matrix(valid.mode,valid.predicted_mode,labels=labels)
    fig,ax=plt.subplots(figsize=(7,5))
    sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',
                xticklabels=labels,yticklabels=labels,ax=ax)
    ax.set_xlabel('Predicted');ax.set_ylabel('Actual (Malin)')
    ax.set_title('Mode detection confusion matrix')
    plt.tight_layout();plt.show()
    print(classification_report(valid.mode,valid.predicted_mode,labels=labels,zero_division=0))

---
## Part 2: Feature Matrix and PCA


In [ ]:
def build_features(df_):
    rows=[]
    for _,row in df_.iterrows():
        counts=[0]*12
        for pc in row.pitch_classes: counts[pc]+=1
        t=sum(counts) or 1;pc_f=[c/t for c in counts]
        ic=Counter(row.intervals);it=sum(ic.values()) or 1
        ivl_f=[ic.get(i,0)/it for i in range(-12,13)]
        degs=[d for d in row.scale_degrees if d!=0]
        rows.append(pc_f+ivl_f+[np.mean(degs) if degs else 0,np.std(degs) if degs else 0])
    cols=[f'pc_{i}' for i in range(12)]+[f'ivl_{i}' for i in range(-12,13)]+['deg_mean','deg_std']
    return pd.DataFrame(rows,index=df_['tune_id'],columns=cols)

X_df=build_features(df)
print(f'Feature matrix: {X_df.shape}')

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X=StandardScaler().fit_transform(X_df.fillna(0))
pca=PCA(n_components=2);coords=pca.fit_transform(X)
pca_df=pd.DataFrame(coords,columns=['PC1','PC2'],index=X_df.index)
if 'mode' in df.columns: pca_df['mode']=df.set_index('tune_id')['mode']

fig,ax=plt.subplots()
for label in pca_df.get('mode',pd.Series()).dropna().unique():
    s=pca_df[pca_df.mode==label]
    ax.scatter(s.PC1,s.PC2,label=label,alpha=0.6,s=30)
ax.legend(fontsize=8)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.set_title('Beregovski corpus in PCA space')
plt.tight_layout();plt.show()

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram,linkage

Z=linkage(X,'ward')
fig,ax=plt.subplots(figsize=(14,5))
dendrogram(Z,ax=ax,truncate_mode='lastp',p=40,leaf_font_size=6)
ax.set_title('Hierarchical clustering (Ward)')
plt.tight_layout();plt.show()

agg=AgglomerativeClustering(n_clusters=4,linkage='ward')
df['cluster']=agg.fit_predict(X)
if 'mode' in df.columns:
    print('Cluster vs mode cross-tab:')
    print(pd.crosstab(df.cluster,df.mode))

---
## Day 7 Exercise: Confusion Matrix as Musical Argument

```{admonition} Exercise
Identify the dominant error in your confusion matrix.
Write 250–300 words explaining it in purely musical terms — not statistical ones.
Argue whether it reveals: (a) a limitation of the method, (b) a genuine musical ambiguity,
or (c) a limitation of Malin's annotation categories.
```


### Your argument

*(250–300 words)*


---
## Project Log — Entry 7

> *Mode detector accuracy: [X]%.*  
> *Dominant confusion: [Y] vs [Z], musically because [explanation].*  
> *Cluster structure [does/does not] match Malin's categories. Most revealing mismatch: [X].*  
> *If I revised my research question it would be...*

*(100–150 words)*


---
```{admonition} Tonight: Final Paper Draft
Assemble log entries 1–7 with connective tissue. Add §8 (What the Method Cannot See).
Target: 1500–2000 words. Notebooks as appendix. Present tomorrow.
```
